In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

# The study scripts (config.py, sim.py) live in ../scripts and hold the shared
# telescope geometry and the danish-based donut simulator we use here.
REPO = Path.cwd().parent
sys.path.insert(0, str(REPO / "scripts"))

import danish  # noqa: E402
import config as C  # noqa: E402
import sim  # noqa: E402

# Match the repo's Computer-Modern figure style (but keep an interactive backend
# so the donuts render inline in the notebook).
mpl.rc_file(REPO / "matplotlibrc")
mpl.rcParams["backend"] = "module://matplotlib_inline.backend_inline"

# --- Shared figure geometry ----------------------------------------------
# The map_circles schematics use a frame half-width of DIAGRAM_L with an outer
# ring radius of 1, so the ring fills 1 / DIAGRAM_L of the panel.  We crop the
# simulated donuts to the same fraction (see show_donut) so the two rows match.
DIAGRAM_L = 1.25  # map_circles frame half-width (outer ring radius = 1)
DONUT_OUTER_PX = 66.7  # measured outer radius of a full-defocus donut, in pixels
ARCSEC_PER_PX = C.PIXEL_SCALE / C.FOCAL_LENGTH * 206265  # plate scale


def simulate_donut(defocus=C.DEFOCUS_Z4, zernikes=None, dx=0.0, dy=0.0, seed=1):
    """Simulate one extra-focal Rubin donut with the repo's danish simulator.

    Uses the same triangle-mesh forward model as the experiments: a defocused
    toy-Rubin paraboloid observed at field center, blurred by ``C.FWHM`` seeing
    and given a little photon noise.  Extra Zernike aberrations imprint their
    characteristic shape on the donut, and a lateral shift ``(dx, dy)`` moves it
    off-centre (the image-plane signature of wavefront tilt).

    The donut is rendered *extra*-focal (negative Z4).  An extra-focal image is
    upright with respect to the pupil, so the pupil-plane schematics drawn by
    ``map_circles`` line up with the donuts directly; the intra-focal image is
    the same donut rotated 180 degrees, which would flip the odd-m modes.

    Parameters
    ----------
    defocus : float, optional
        Magnitude of the Z4 defocus, in meters (sets the donut diameter).
    zernikes : dict, optional
        Extra aberrations to add, as ``{noll_index: coefficient_in_meters}``
        (e.g. ``{5: 7e-6}`` for astigmatism).
    dx, dy : float, optional
        Centroid offset in arcseconds (the tilt signature).
    seed : int, optional
        Photon-noise seed.

    Returns
    -------
    ndarray
        The simulated donut image, shape ``(C.NPIX, C.NPIX)``.
    """
    zernikes = zernikes or {}
    jmax = max([C.JMAX, *zernikes.keys()])
    z_ref = np.zeros(jmax + 1)
    z_ref[4] = -defocus  # negative Z4 -> extra-focal, upright pupil mapping
    for noll, coeff in zernikes.items():
        z_ref[noll] += coeff

    model = danish.SingleDonutModel(
        sim.make_factory(surface_brightness=True),
        z_ref=z_ref,
        z_terms=[],
        thx=0.0,
        thy=0.0,
        bkg_order=-1,
        seed=seed,
        npix=C.NPIX,
    )
    # Normalize to a fixed total flux, then render with seeing + sky noise.
    base = model.model(1.0, 0.0, 0.0, C.FWHM, [], sky_level=None)
    flux = C.FLUX / base.sum()
    return model.model(flux, dx, dy, C.FWHM, [], sky_level=C.SKY_LEVEL)


def show_donut(ax, img):
    """Show a donut on ``ax``, cropped to match the schematic framing.

    Crops so a full-defocus ring fills the same fraction of the panel as the
    map_circles schematic ring (outer radius / half-width = 1 / DIAGRAM_L), then
    hides the axis ticks.
    """
    crop = C.NPIX // 2 - int(round(DONUT_OUTER_PX * DIAGRAM_L))
    ax.imshow(img[crop:-crop, crop:-crop], origin="lower")
    ax.set(xticks=[], yticks=[])


In [ ]:
def map_circles(A, m, displacement, N=5, ax=None, center=True, **kwargs):
    """Map concentric pupil circles through a Cartesian displacement field.

    ``displacement(rho, theta, m)`` returns ``(dx, dy)`` in outer-pupil-radius
    units.  Polar gradient formulas can be converted with
    ``polar_displacement_to_xy`` below.
    """
    kwargs = {"c": "C1", "lw": 0.5, "zorder": 10} | kwargs
    if ax is None:
        fig, ax = plt.subplots()

    theta = np.linspace(0, 2 * np.pi, 10_000)
    for r in np.linspace(C.EPS_RUBIN, 1, N):
        rho = r * np.ones_like(theta)
        x0 = rho * np.cos(theta)
        y0 = rho * np.sin(theta)
        dx, dy = displacement(rho, theta, m)
        ax.plot(x0 + A * dx, y0 + A * dy, **kwargs)

    if center:
        x0 = np.mean(ax.get_xlim())
        y0 = np.mean(ax.get_ylim())
    else:
        x0, y0 = 0, 0
    ax.set(
        xlim=(x0 - DIAGRAM_L, x0 + DIAGRAM_L),
        ylim=(y0 - DIAGRAM_L, y0 + DIAGRAM_L),
        aspect="equal",
        xticks=[],
        yticks=[],
    )


def polar_displacement_to_xy(rho, theta, drho, dtheta):
    """Convert radial/tangential displacement components to Cartesian dx, dy."""
    dx = drho * np.cos(theta) - dtheta * np.sin(theta)
    dy = drho * np.sin(theta) + dtheta * np.cos(theta)
    return dx, dy


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(3.5, 3.8), constrained_layout=True, dpi=150)

axes[0, 0].set_title("Defocus\n$\\nu = 0,\\, m=0$")
axes[0, 1].set_title("Tilt\n$\\nu = 0,\\, m=1$")

# --- Top row: wavefront-circle schematics --------------------------------
# Defocus (m=0): a purely radial rescaling of the pupil circles (here a 25%
# shrink, A=0.25).  Tilt (m=1): a rigid lateral shift, no size change.  The
# tilt perturbation below shifts every circle straight down by 0.18 of the
# outer radius.
DEFOCUS_SHRINK = 0.25  # radial shrink used in the defocus schematic
TILT_SHIFT = 0.18  # downward shift of the tilt schematic, in outer-radius units

defocus_displacement = lambda rho, theta, m: polar_displacement_to_xy(
    rho, theta, -rho, 0
)
map_circles(DEFOCUS_SHRINK, 0, defocus_displacement, ax=axes[0, 0])

tilt_displacement = lambda rho, theta, m: (np.zeros_like(rho), -np.ones_like(rho))
map_circles(TILT_SHIFT, 0, tilt_displacement, ax=axes[0, 1], center=False)

# --- Bottom row: simulated extra-focal donuts ----------------------------
# Each donut shows its mode's signature.  Defocus sets the ring *size*: the
# defocus panel uses less absolute defocus so its ring is smaller (shrunk by the
# same (1 - DEFOCUS_SHRINK) factor as the schematic above), shown centred.  Tilt
# leaves the size unchanged but *shifts* the ring straight down, matching the
# tilt schematic (its circles shift by TILT_SHIFT of the outer radius in -y).
donut_defocus = simulate_donut(defocus=(1 - DEFOCUS_SHRINK) * C.DEFOCUS_Z4)
donut_tilt = simulate_donut(
    dy=-TILT_SHIFT * DONUT_OUTER_PX * ARCSEC_PER_PX
)

show_donut(axes[1, 0], donut_defocus)
show_donut(axes[1, 1], donut_tilt)

fig.savefig("../figures/abberations_defocus_tilt.pdf", bbox_inches="tight")


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(7, 3.8), constrained_layout=True, dpi=120)

axes[0, 0].set_title("Astigmatism\n$\\nu = 0,\\, m=2$")
axes[0, 1].set_title("Trefoil\n$\\nu = 0,\\, m=3$")
axes[0, 2].set_title("Quadrafoil\n$\\nu = 0,\\, m=4$")
axes[0, 3].set_title("Pentafoil\n$\\nu = 0,\\, m=5$")

# --- Top row: wavefront-circle schematics --------------------------------
# A donut is the pupil imaged through the wavefront slope: a ray leaving pupil
# point (u, v) lands displaced by the transverse ray aberration, -grad(W).  For
# an m-foil W = rho^m sin(m theta) the Cartesian gradient is
#   grad(W) = m rho^(m-1) (sin((m-1) theta), cos((m-1) theta)),
# so we displace each pupil circle by -grad(W).  Because the donuts below are
# extra-focal (upright) images, this one mapping matches every mode -- no
# per-mode sign flips needed (see simulate_donut for the extra-focal choice).
mfoil_displacement = lambda rho, theta, m: (
    -m * rho**(m - 1) * np.sin((m - 1) * theta),
    -m * rho**(m - 1) * np.cos((m - 1) * theta),
)

map_circles(0.070, 2, mfoil_displacement, ax=axes[0, 0])
map_circles(0.020, 3, mfoil_displacement, ax=axes[0, 1])
map_circles(0.010, 4, mfoil_displacement, ax=axes[0, 2])
map_circles(0.007, 5, mfoil_displacement, ax=axes[0, 3])

# --- Bottom row: simulated extra-focal donuts ----------------------------
# One donut per mode, each with the lowest Noll index of that azimuthal order
# (astigmatism Z5, trefoil Z9, quadrafoil Z15, pentafoil Z21) added to the
# baseline defocus.  The aberration imprints its m-fold shape on the donut.
show_donut(axes[1, 0], simulate_donut(zernikes={5: 7e-6}))
show_donut(axes[1, 1], simulate_donut(zernikes={9: 2e-6}))
show_donut(axes[1, 2], simulate_donut(zernikes={15: 1e-6}))
show_donut(axes[1, 3], simulate_donut(zernikes={21: 0.5e-6}))

fig.savefig("../figures/abberations_astig_mfoil.pdf", bbox_inches="tight")


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(3.5, 3.8), constrained_layout=True, dpi=150)

axes[0, 0].set_title("Spherical\n$\\nu = 1,\\, m=0$")
axes[0, 1].set_title("Coma\n$\\nu = 1,\\, m=1$")

# --- Top row: wavefront-circle schematics --------------------------------
# Positive Z11 has W ~ rho^4 - (1 + eps^2) rho^2.  The extra-focal donut is
# upright with respect to the pupil, so the plotted displacement is -grad(W):
# inner and outer rings move toward each other.
EPS = C.EPS_RUBIN
spherical_displacement = lambda rho, theta, m: polar_displacement_to_xy(
    rho, theta, -2 * rho**3 + (1 + EPS**2) * rho, 0
)
map_circles(0.17, 0, spherical_displacement, ax=axes[0, 0])

# Positive Z7 sine coma, dropping the pure-tilt term.  The dominant term shifts
# each ring by an amount proportional to rho^2, decentering the hole relative to
# the outer edge.
COMA_A = 1 + EPS**2
coma_displacement = lambda rho, theta, m: (
    -3 * COMA_A * rho**2 * np.sin(2 * theta),
    3 * COMA_A * rho**2 * np.cos(2 * theta) - 6 * COMA_A * rho**2,
)
map_circles(0.03, 0, coma_displacement, ax=axes[0, 1])

# --- Bottom row: simulated extra-focal donuts ----------------------------
# The image panels use the corresponding positive sine-side Noll coefficients:
# spherical Z11 and coma Z7.
show_donut(axes[1, 0], simulate_donut(zernikes={11: 0.2e-6}))
show_donut(axes[1, 1], simulate_donut(zernikes={7: 2e-6}))

fig.savefig("../figures/abberations_spherical_coma.pdf", bbox_inches="tight")


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(7, 3.8), constrained_layout=True, dpi=120)

axes[0, 0].set_title("2nd Astigmatism\n$\\nu = 1,\\, m=2$")
axes[0, 1].set_title("2nd Trefoil\n$\\nu = 1,\\, m=3$")
axes[0, 2].set_title("2nd Quadrafoil\n$\\nu = 1,\\, m=4$")
axes[0, 3].set_title("2nd Pentafoil\n$\\nu = 1,\\, m=5$")

# --- Top row: wavefront-circle schematics --------------------------------
# For the secondary m-foils,
#   W ~ [(m + 2) rho^(m + 2) - (m + 1) A_m rho^m] sin(m theta),
# with A_m set by annular orthogonality.  We again plot -grad(W), converted
# from polar radial/tangential components into Cartesian displacements.
EPS = C.EPS_RUBIN


def secondary_mfoil_displacement(rho, theta, m):
    annular_a = (1 - EPS ** (2 * (m + 2))) / (1 - EPS ** (2 * (m + 1)))
    drho = (
        m * (m + 1) * annular_a * rho ** (m - 1)
        - (m + 2) ** 2 * rho ** (m + 1)
    ) * np.sin(m * theta)
    dtheta = (
        m * (m + 1) * annular_a * rho ** (m - 1)
        - m * (m + 2) * rho ** (m + 1)
    ) * np.cos(m * theta)
    return polar_displacement_to_xy(rho, theta, drho, dtheta)


map_circles(0.018, 2, secondary_mfoil_displacement, ax=axes[0, 0])
map_circles(0.007, 3, secondary_mfoil_displacement, ax=axes[0, 1])
map_circles(0.004, 4, secondary_mfoil_displacement, ax=axes[0, 2])
map_circles(0.0025, 5, secondary_mfoil_displacement, ax=axes[0, 3])

# --- Bottom row: simulated extra-focal donuts ----------------------------
# These are the positive sine-side secondary m-foil modes.  Z25 and Z33 are
# display-only higher-Noll modes, so simulate_donut pads the Zernike vector.
show_donut(axes[1, 0], simulate_donut(zernikes={13: 0.75e-6}))
show_donut(axes[1, 1], simulate_donut(zernikes={19: 0.4e-6}))
show_donut(axes[1, 2], simulate_donut(zernikes={25: 0.22e-6}))
show_donut(axes[1, 3], simulate_donut(zernikes={33: 0.12e-6}))

fig.savefig("../figures/abberations_2nd_astig_mfoil.pdf", bbox_inches="tight")


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(3.5, 3.8), constrained_layout=True, dpi=150)

axes[0, 0].set_title("2nd Spherical\n$\\nu = 2,\\, m=0$")
axes[0, 1].set_title("2nd Coma\n$\\nu = 2,\\, m=1$")

# --- Top row: wavefront-circle schematics --------------------------------
# Secondary spherical (Z22): W has rho^6, rho^4, and rho^2 terms.  The plotted
# displacement is -grad(W), with the arbitrary overall normalization absorbed
# into the display amplitude below.
EPS = C.EPS_RUBIN
secondary_spherical_displacement = lambda rho, theta, m: polar_displacement_to_xy(
    rho,
    theta,
    -5 * rho**5
    + 5 * (1 + EPS**2) * rho**3
    - (1 + 3 * EPS**2 + EPS**4) * rho,
    0,
)
map_circles(0.17, 0, secondary_spherical_displacement, ax=axes[0, 0])

# Secondary sine coma (Z17), including the annular low-order term.  That term
# is a global tilt; map_circles recenters the view, leaving the relative
# rho-dependent distortion that matters for the donut shape.
SC_A = 10 * (1 + 4 * EPS**2 + EPS**4)
SC_B = 12 * (1 + 4 * EPS**2 + 4 * EPS**4 + EPS**6)
SC_D = 3 * (1 + 4 * EPS**2 + 10 * EPS**4 + 4 * EPS**6 + EPS**8)


def secondary_coma_displacement(rho, theta, m):
    drho = -(5 * SC_A * rho**4 - 3 * SC_B * rho**2 + SC_D) * np.sin(theta)
    dtheta = -(SC_A * rho**4 - SC_B * rho**2 + SC_D) * np.cos(theta)
    return polar_displacement_to_xy(rho, theta, drho, dtheta)


map_circles(0.003, 0, secondary_coma_displacement, ax=axes[0, 1])

# --- Bottom row: simulated extra-focal donuts ----------------------------
# The image panels use the corresponding positive Noll coefficients: secondary
# spherical Z22 and sine secondary coma Z17.
show_donut(axes[1, 0], simulate_donut(zernikes={22: 0.05e-6}))
show_donut(axes[1, 1], simulate_donut(zernikes={17: 0.2e-6}))

fig.savefig("../figures/abberations_2nd_spherical_coma.pdf", bbox_inches="tight")
